# 🔥 Tamil Hate Speech Detection — Model Training (Colab GPU)

This notebook fine-tunes **MuRIL**, **XLM-RoBERTa**, and **mBERT** on the Tamil OffensEval Dravidian dataset.

**Requirements:** Run this on Google Colab with GPU runtime.

Go to `Runtime > Change runtime type > T4 GPU`

## 1. Setup

In [ ]:
# Install dependencies
!pip install -q datasets transformers torch scikit-learn pandas numpy matplotlib seaborn lime accelerate tqdm

In [ ]:
# Set project directory (no Drive mount needed)
import os

PROJECT_DIR = '/content/hate_speech'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)

# Create output directories
for d in ['outputs/models', 'outputs/evaluation', 'outputs/analysis/lime_explanations']:
    os.makedirs(d, exist_ok=True)

print(f'Working directory: {os.getcwd()}')

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Load & Preprocess Data

In [ ]:
import re
import pandas as pd
import numpy as np
from datasets import load_dataset, Dataset, DatasetDict

# Label definitions
LABEL_NAMES = [
    'Not_offensive', 'Offensive_Untargeted', 'Offensive_Targeted_Individual',
    'Offensive_Targeted_Group', 'Offensive_Targeted_Other', 'not-Tamil'
]
LABEL2ID = {name: idx for idx, name in enumerate(LABEL_NAMES)}
ID2LABEL = {idx: name for idx, name in enumerate(LABEL_NAMES)}
NUM_LABELS = len(LABEL_NAMES)

def clean_text(text):
    if not isinstance(text, str): return ''
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'[^\w\s\u0B80-\u0BFF]', ' ', text)
    text = ''.join(c.lower() if 'A' <= c <= 'Z' else c for c in text)
    return re.sub(r'\s+', ' ', text).strip()

# Load dataset
print('Loading dataset...')
dataset = load_dataset('community-datasets/offenseval_dravidian', 'tamil')
train_data, val_data = dataset['train'], dataset['validation']
print(f'Train: {len(train_data):,} | Val: {len(val_data):,}')

# Clean
train_texts = [clean_text(t) for t in train_data['text']]
val_texts = [clean_text(t) for t in val_data['text']]

train_df = pd.DataFrame({'text': train_texts, 'label': train_data['label']})
val_df = pd.DataFrame({'text': val_texts, 'label': val_data['label']})
train_df = train_df[train_df['text'].str.strip().str.len() > 0].reset_index(drop=True)
val_df = val_df[val_df['text'].str.strip().str.len() > 0].reset_index(drop=True)

# Split val 50/50 into val and test
val_df = val_df.sample(frac=1, random_state=42).reset_index(drop=True)
split = len(val_df) // 2
test_df = val_df[split:].reset_index(drop=True)
val_df = val_df[:split].reset_index(drop=True)

hf_dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df),
    'validation': Dataset.from_pandas(val_df),
    'test': Dataset.from_pandas(test_df),
})

print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')

## 3. Training Utilities

In [ ]:
from torch import nn
from transformers import Trainer, TrainingArguments, AutoTokenizer, AutoModelForSequenceClassification, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    p_m, r_m, f1_m, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    p_w, r_w, f1_w, _ = precision_recall_fscore_support(labels, preds, average='weighted', zero_division=0)
    return {'accuracy': acc, 'f1_macro': f1_m, 'f1_weighted': f1_w, 'precision_macro': p_m, 'recall_macro': r_m, 'precision_weighted': p_w, 'recall_weighted': r_w}

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = torch.tensor(class_weights, dtype=torch.float32) if class_weights is not None else None

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        if self.class_weights is not None:
            loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        else:
            loss_fct = nn.CrossEntropyLoss()
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_class_weights(labels, num_classes=NUM_LABELS):
    counts = np.bincount(labels, minlength=num_classes).astype(float)
    counts = np.maximum(counts, 1.0)
    return (counts.sum() / (num_classes * counts)).tolist()

## 4. Train All Models
This trains MuRIL, XLM-RoBERTa, and mBERT sequentially (~50 min total on T4)

In [ ]:
import json

MODEL_CONFIGS = {
    'muril': {'name': 'MuRIL', 'hf_id': 'google/muril-base-cased'},
    'xlm-roberta': {'name': 'XLM-RoBERTa', 'hf_id': 'xlm-roberta-base'},
    'mbert': {'name': 'mBERT', 'hf_id': 'bert-base-multilingual-cased'},
}

all_histories = {}

for model_key, config in MODEL_CONFIGS.items():
    print(f'\n{"="*60}')
    print(f'  Training: {config["name"]} ({config["hf_id"]})')
    print(f'{"="*60}')

    output_dir = f'outputs/models/{model_key}'
    os.makedirs(output_dir, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(config['hf_id'])

    def tokenize_fn(examples):
        return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)

    train_tok = hf_dataset['train'].map(tokenize_fn, batched=True, remove_columns=['text'])
    val_tok = hf_dataset['validation'].map(tokenize_fn, batched=True, remove_columns=['text'])
    train_tok = train_tok.rename_column('label', 'labels').with_format('torch')
    val_tok = val_tok.rename_column('label', 'labels').with_format('torch')

    model = AutoModelForSequenceClassification.from_pretrained(config['hf_id'], num_labels=NUM_LABELS, id2label=ID2LABEL, label2id=LABEL2ID)

    class_weights = compute_class_weights(np.array(hf_dataset['train']['label']))

    training_args = TrainingArguments(
        output_dir=output_dir, num_train_epochs=5,
        per_device_train_batch_size=16, per_device_eval_batch_size=32,
        gradient_accumulation_steps=2, learning_rate=2e-5,
        weight_decay=0.01, warmup_ratio=0.1, fp16=True,
        eval_strategy='epoch', save_strategy='epoch',
        load_best_model_at_end=True, metric_for_best_model='f1_weighted',
        greater_is_better=True, save_total_limit=2,
        logging_steps=100, report_to='none', seed=42,
    )

    trainer = WeightedTrainer(
        class_weights=class_weights, model=model, args=training_args,
        train_dataset=train_tok, eval_dataset=val_tok,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    train_result = trainer.train()

    # Save best model + tokenizer
    best_dir = f'{output_dir}/best_model'
    trainer.save_model(best_dir)
    tokenizer.save_pretrained(best_dir)

    # Evaluate on validation
    eval_results = trainer.evaluate()

    all_histories[model_key] = {
        'model': config['name'], 'hf_id': config['hf_id'],
        'best_f1_weighted': trainer.state.best_metric,
        'train_runtime': train_result.metrics.get('train_runtime', 0),
        'eval_metrics': eval_results,
    }

    print(f'\n  Best F1-Weighted: {trainer.state.best_metric:.4f}')

    # === CLEANUP: Remove intermediate checkpoints to save disk/download space ===
    import shutil
    for item in os.listdir(output_dir):
        item_path = os.path.join(output_dir, item)
        if item.startswith('checkpoint-') and os.path.isdir(item_path):
            shutil.rmtree(item_path)
            print(f'  Cleaned up: {item}')

    # Free GPU memory
    del model, trainer
    torch.cuda.empty_cache()

# Save training summary
with open('outputs/models/training_summary.json', 'w') as f:
    json.dump(all_histories, f, indent=2)

print(f'\n\n=== ALL MODELS TRAINED ===')
for k, h in all_histories.items():
    print(f'  {h["model"]}: F1-W = {h["best_f1_weighted"]:.4f}')

## 5. Evaluate on Test Set

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

SHORT_LABELS = ['Not Off.', 'Off. Untarg.', 'Off. Indiv.', 'Off. Group', 'Off. Other', 'not-Tamil']

test_texts = hf_dataset['test']['text']
test_labels = np.array(hf_dataset['test']['label'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

all_metrics = {}
all_preds = {}

for model_key in MODEL_CONFIGS:
    model_path = f'outputs/models/{model_key}/best_model'
    if not os.path.exists(model_path):
        print(f'Skipping {model_key} - not found')
        continue

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)
    model.eval()

    preds = []
    for i in range(0, len(test_texts), 32):
        batch = list(test_texts[i:i+32])
        enc = tokenizer(batch, padding='max_length', truncation=True, max_length=128, return_tensors='pt').to(device)
        with torch.no_grad():
            preds.extend(model(**enc).logits.argmax(-1).cpu().tolist())

    preds = np.array(preds)
    all_preds[model_key] = preds

    report = classification_report(test_labels, preds, target_names=LABEL_NAMES, output_dict=True, zero_division=0)
    all_metrics[model_key] = {
        'accuracy': report['accuracy'],
        'f1_weighted': report['weighted avg']['f1-score'],
        'f1_macro': report['macro avg']['f1-score'],
        'precision_weighted': report['weighted avg']['precision'],
        'recall_weighted': report['weighted avg']['recall'],
    }

    # Per-class F1
    for i, name in enumerate(LABEL_NAMES):
        all_metrics[model_key][f'f1_{name}'] = report.get(name, {}).get('f1-score', 0)

    print(f'\n{"="*50}')
    print(f'  {MODEL_CONFIGS[model_key]["name"]}')
    print(f'{"="*50}')
    print(classification_report(test_labels, preds, target_names=LABEL_NAMES, zero_division=0))

    # Confusion matrix
    cm = confusion_matrix(test_labels, preds)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd', xticklabels=SHORT_LABELS, yticklabels=SHORT_LABELS, ax=ax)
    ax.set_title(f'{MODEL_CONFIGS[model_key]["name"]} - Confusion Matrix')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    plt.tight_layout()
    plt.savefig(f'outputs/evaluation/{model_key}_confusion_matrix.png', dpi=150)
    plt.show()

    del model
    torch.cuda.empty_cache()

In [ ]:
# Print comparison table
print('\n' + '='*70)
print('  MODEL COMPARISON (TEST SET)')
print('='*70)
print(f'{"Model":<20s} {"Accuracy":>10s} {"F1-W":>10s} {"F1-M":>10s} {"Prec-W":>10s} {"Rec-W":>10s}')
print('-'*70)
for mk, m in all_metrics.items():
    print(f'{MODEL_CONFIGS[mk]["name"]:<20s} {m["accuracy"]:>10.4f} {m["f1_weighted"]:>10.4f} {m["f1_macro"]:>10.4f} {m["precision_weighted"]:>10.4f} {m["recall_weighted"]:>10.4f}')

best_key = max(all_metrics, key=lambda k: all_metrics[k]['f1_weighted'])
print(f'\nBest model: {MODEL_CONFIGS[best_key]["name"]} (F1-W: {all_metrics[best_key]["f1_weighted"]:.4f})')

# Save everything
comp_df = pd.DataFrame(all_metrics).T
comp_df.to_csv('outputs/evaluation/comparison_metrics.csv')

pred_df = pd.DataFrame({'text': test_texts, 'true_label': test_labels, 'true_label_name': [LABEL_NAMES[l] for l in test_labels]})
for mk in all_preds:
    pred_df[f'pred_{mk}'] = all_preds[mk]
    pred_df[f'pred_{mk}_name'] = [LABEL_NAMES[p] for p in all_preds[mk]]
pred_df.to_csv('outputs/evaluation/all_predictions.csv', index=False)

with open('outputs/evaluation/comparison_metrics.json', 'w') as f:
    json.dump(all_metrics, f, indent=2)

print('\nAll results saved to outputs/evaluation/')

## 6. Model Comparison Chart

In [ ]:
metric_names = ['Accuracy', 'F1 (Weighted)', 'F1 (Macro)', 'Precision (W)', 'Recall (W)']
metric_keys = ['accuracy', 'f1_weighted', 'f1_macro', 'precision_weighted', 'recall_weighted']

fig, ax = plt.subplots(figsize=(14, 7))
x = np.arange(len(metric_names))
width = 0.25
colors = ['#00d2ff', '#f5af19', '#fc466b']

for i, (mk, metrics) in enumerate(all_metrics.items()):
    values = [metrics.get(k, 0) for k in metric_keys]
    bars = ax.bar(x + i * width, values, width, label=MODEL_CONFIGS[mk]['name'], color=colors[i], edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, f'{val:.3f}', ha='center', va='bottom', fontsize=9)

ax.set_ylabel('Score')
ax.set_title('Model Comparison - Test Set Performance')
ax.set_xticks(x + width)
ax.set_xticklabels(metric_names)
ax.legend()
ax.set_ylim(0, 1.1)
ax.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.savefig('outputs/evaluation/model_comparison_table.png', dpi=150)
plt.show()

## 7. Error Analysis

In [ ]:
# Use best model for error analysis
best_pred_col = f'pred_{best_key}'
best_pred_name = f'pred_{best_key}_name'

pred_df['correct'] = pred_df['true_label'] == pred_df[best_pred_col]
total = len(pred_df)
correct = pred_df['correct'].sum()
incorrect = total - correct

print(f'Best model: {MODEL_CONFIGS[best_key]["name"]}')
print(f'Correct:   {correct:,} ({correct/total*100:.1f}%)')
print(f'Incorrect: {incorrect:,} ({incorrect/total*100:.1f}%)')

misclassified = pred_df[~pred_df['correct']].copy()

# Dangerous false negatives
false_neg = misclassified[(misclassified['true_label'] != 0) & (misclassified['true_label'] != 5) & (misclassified[best_pred_col] == 0)]
false_pos = misclassified[(misclassified['true_label'] == 0) & (misclassified[best_pred_col] != 0) & (misclassified[best_pred_col] != 5)]

print(f'\nDangerous false negatives (offensive -> not_offensive): {len(false_neg)}')
print(f'False positives (not_offensive -> offensive): {len(false_pos)}')

print('\n--- Sample False Negatives ---')
for _, row in false_neg.head(5).iterrows():
    print(f'  TRUE: {row["true_label_name"]} | PRED: {row[best_pred_name]}')
    print(f'  TEXT: {str(row["text"])[:100]}')
    print()

# Error rate by class
print('\n--- Error Rate by Class ---')
for i, name in enumerate(LABEL_NAMES):
    class_total = (pred_df['true_label'] == i).sum()
    class_errors = ((pred_df['true_label'] == i) & ~pred_df['correct']).sum()
    rate = class_errors / class_total * 100 if class_total > 0 else 0
    print(f'  {name:<35s} {rate:5.1f}% ({class_errors}/{class_total})')

# Save error analysis
misclassified.to_csv('outputs/analysis/misclassified_samples.csv', index=False)

error_summary = {
    'best_model': best_key,
    'total': total, 'correct': int(correct), 'incorrect': int(incorrect),
    'false_negatives': len(false_neg), 'false_positives': len(false_pos),
}
with open('outputs/analysis/error_summary.json', 'w') as f:
    json.dump(error_summary, f, indent=2)

print('\nError analysis saved to outputs/analysis/')

## 8. LIME Explainability

In [ ]:
from lime.lime_text import LimeTextExplainer

# Load best model for LIME
best_model_path = f'outputs/models/{best_key}/best_model'
tokenizer = AutoTokenizer.from_pretrained(best_model_path)
model = AutoModelForSequenceClassification.from_pretrained(best_model_path).to(device)
model.eval()

def predict_proba(texts):
    all_probs = []
    for i in range(0, len(texts), 16):
        batch = list(texts[i:i+16])
        enc = tokenizer(batch, padding='max_length', truncation=True, max_length=128, return_tensors='pt').to(device)
        with torch.no_grad():
            logits = model(**enc).logits
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
            all_probs.append(probs)
    return np.vstack(all_probs)

explainer = LimeTextExplainer(class_names=LABEL_NAMES, verbose=False)

# Explain 5 misclassified + 3 correct offensive samples
samples = pd.concat([
    misclassified.head(5),
    pred_df[(pred_df['correct']) & (pred_df['true_label'].isin([1,2,3,4]))].head(3)
]).reset_index(drop=True)

print(f'Generating LIME explanations for {len(samples)} samples...\n')

explanation_summaries = []
for idx, row in samples.iterrows():
    text = str(row['text'])
    is_correct = row['correct']
    pred_label = int(row[best_pred_col])
    print(f'[{idx+1}/{len(samples)}] {"CORRECT" if is_correct else "WRONG"} | True: {row["true_label_name"]} | Pred: {row[best_pred_name]}')
    print(f'  {text[:80]}')

    try:
        exp = explainer.explain_instance(text, predict_proba, num_features=10, num_samples=500)
        exp.save_to_file(f'outputs/analysis/lime_explanations/sample_{idx:03d}.html')
        top_features = exp.as_list(label=pred_label)
        print(f'  Top features: {top_features[:5]}')
        explanation_summaries.append({
            'sample_idx': idx, 'text': text[:200],
            'true_label': row['true_label_name'], 'predicted_label': row[best_pred_name],
            'correct': is_correct, 'top_features': str(top_features[:10]),
        })
    except Exception as e:
        print(f'  Error: {e}')
    print()

pd.DataFrame(explanation_summaries).to_csv('outputs/analysis/lime_explanations/explanation_summary.csv', index=False)
del model
torch.cuda.empty_cache()
print('LIME explanations saved to outputs/analysis/lime_explanations/')

## 9. Verify & Download Results

This section:
1. Removes bulky optimizer/checkpoint files (not needed for inference)
2. Verifies all expected output files exist
3. Downloads results as separate per-model zips + one results zip

In [ ]:
import shutil, glob

# ============================================================
# STEP 1: Clean up — remove optimizer.pt, training_states, and
#          intermediate checkpoints to shrink download size
# ============================================================
print('=== CLEANUP ===')
cleanup_patterns = [
    'outputs/models/**/optimizer.pt',
    'outputs/models/**/training_args.bin',
    'outputs/models/**/trainer_state.json',
    'outputs/models/**/rng_state*.pth',
    'outputs/models/**/scheduler.pt',
]
for pattern in cleanup_patterns:
    for f in glob.glob(pattern, recursive=True):
        # Only delete from checkpoint dirs, NOT from best_model
        if 'best_model' not in f:
            size_mb = os.path.getsize(f) / 1e6
            os.remove(f)
            print(f'  Deleted: {f} ({size_mb:.1f} MB)')

# Remove any remaining checkpoint-* directories
for model_key in MODEL_CONFIGS:
    model_dir = f'outputs/models/{model_key}'
    if os.path.exists(model_dir):
        for item in os.listdir(model_dir):
            if item.startswith('checkpoint-'):
                path = os.path.join(model_dir, item)
                shutil.rmtree(path)
                print(f'  Removed checkpoint dir: {path}')

# ============================================================
# STEP 2: Verify all expected files exist
# ============================================================
print('\n=== VERIFICATION ===')
expected_files = {
    'Models': [],
    'Evaluation': [
        'outputs/evaluation/comparison_metrics.json',
        'outputs/evaluation/comparison_metrics.csv',
        'outputs/evaluation/all_predictions.csv',
        'outputs/evaluation/model_comparison_table.png',
    ],
    'Analysis': [
        'outputs/analysis/error_summary.json',
        'outputs/analysis/misclassified_samples.csv',
    ],
}

# Add model files
for mk in MODEL_CONFIGS:
    best_dir = f'outputs/models/{mk}/best_model'
    expected_files['Models'].append(best_dir)
    expected_files['Evaluation'].append(f'outputs/evaluation/{mk}_confusion_matrix.png')

all_ok = True
for category, files in expected_files.items():
    print(f'\n  {category}:')
    for fp in files:
        exists = os.path.exists(fp)
        status = 'OK' if exists else 'MISSING'
        icon = '\u2713' if exists else '\u2717'
        print(f'    {icon} {fp} [{status}]')
        if not exists:
            all_ok = False

# Show sizes
print('\n=== DIRECTORY SIZES ===')
for d in ['outputs/models', 'outputs/evaluation', 'outputs/analysis']:
    if os.path.exists(d):
        total = sum(os.path.getsize(os.path.join(dp, f)) for dp, dn, fns in os.walk(d) for f in fns)
        print(f'  {d}: {total/1e6:.1f} MB')

if all_ok:
    print('\n\u2705 All files verified!')
else:
    print('\n\u26a0 Some files are missing — check the cells above for errors.')

In [ ]:
# ============================================================
# STEP 3: Download — split into manageable zips
# ============================================================
from google.colab import files

# 3a. Download each model separately (these are the big files)
for model_key in MODEL_CONFIGS:
    best_dir = f'outputs/models/{model_key}/best_model'
    if os.path.exists(best_dir):
        zip_name = f'/content/{model_key}_best_model'
        shutil.make_archive(zip_name, 'zip', best_dir)
        size_mb = os.path.getsize(f'{zip_name}.zip') / 1e6
        print(f'Downloading {model_key}_best_model.zip ({size_mb:.1f} MB)...')
        files.download(f'{zip_name}.zip')
    else:
        print(f'Skipping {model_key} — best_model not found')

In [ ]:
# 3b. Download evaluation + analysis results (small files)
!cd /content/hate_speech && zip -r /content/evaluation_results.zip \
    outputs/evaluation/ \
    outputs/analysis/ \
    outputs/models/training_summary.json

size_mb = os.path.getsize('/content/evaluation_results.zip') / 1e6
print(f'\nDownloading evaluation_results.zip ({size_mb:.1f} MB)...')
files.download('/content/evaluation_results.zip')

## 10. Local Setup Instructions

After downloading, extract the files on your local machine:

```bash
# Extract each model into the correct directory
# On Windows (PowerShell):
Expand-Archive muril_best_model.zip     -DestinationPath e:\hate_speech\outputs\models\muril\best_model
Expand-Archive xlm-roberta_best_model.zip -DestinationPath e:\hate_speech\outputs\models\xlm-roberta\best_model
Expand-Archive mbert_best_model.zip     -DestinationPath e:\hate_speech\outputs\models\mbert\best_model

# Extract evaluation results
Expand-Archive evaluation_results.zip   -DestinationPath e:\hate_speech\
```

Then you can run the local evaluation and paper generation scripts.